# Imports

In [1]:
%%capture
!pip install segmentation-models-pytorch
!pip install torchinfo

In [2]:
# Data handling
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Torch
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp
from torchinfo import summary

# os
import os

# Path
from pathlib import Path

# tqdm
from tqdm.auto import tqdm

# warnings
import warnings
warnings.filterwarnings("ignore")

import random as rnd

import shutil

from glob import iglob, glob
from itertools import chain

In [3]:
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

# Num of samples, that will be used for train out model
K_SAMPLES_TRAIN = 5

NUM_CLASSES = 2

# CUDA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataloaders

## Download Malaria

In [4]:
!wget "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/5bf2kmwvfn-1.zip" -O MSeg.zip

--2024-08-08 07:58:06--  https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/5bf2kmwvfn-1.zip
Resolving prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com)... 52.92.18.90, 3.5.64.173, 52.218.121.250, ...
Connecting to prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com (prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com)|52.92.18.90|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1769427652 (1.6G) [application/octet-stream]
Saving to: ‘MSeg.zip’

MSeg.zip            100%[===================>]   1.65G  29.8MB/s    in 58s     

2024-08-08 07:59:04 (29.2 MB/s) - ‘MSeg.zip’ saved [1769427652/1769427652]



In [5]:
!mkdir /content/mseg

In [6]:
!unzip /content/MSeg.zip -d /content/mseg/

Archive:  /content/MSeg.zip
   creating: /content/mseg/Giemsa stained images/
  inflating: /content/mseg/Giemsa stained images/Trip 038 Day 1 01-12-05 Image 10 add_5.png  
  inflating: /content/mseg/Giemsa stained images/Trip 022 Day 1 08-11-05 Image 11 add_14.png  
  inflating: /content/mseg/Giemsa stained images/Trip 038 Day 1 01-12-05 Image 30 add_12.png  
  inflating: /content/mseg/Giemsa stained images/Trip 029 Day 1 22-11-05 Image 10 add_12.png  
  inflating: /content/mseg/Giemsa stained images/Trip 062 Day 1 02-11-05 Image 11 add_16.png  
  inflating: /content/mseg/Giemsa stained images/Trip 038 Day 1 01-12-05 Image 10 add_14.png  
  inflating: /content/mseg/Giemsa stained images/Trip 022 Day 1 08-11-05 Image 11 add_5.png  
  inflating: /content/mseg/Giemsa stained images/Trip 807 Day 1 07-12-05 Image 2_3.png  
  inflating: /content/mseg/Giemsa stained images/Trip 029 Day 1 22-11-05 Image 11 add_2.png  
  inflating: /content/mseg/Giemsa stained images/Trip 043 Day 2 24-10-05 Ima

In [7]:
!rm /content/MSeg.zip

## Defining train for meta-test

In [8]:
def get_mseg_train():
    global K_SAMPLES_TRAIN

    rnd.seed(102)

    train_images = rnd.sample(glob("/content/mseg/Giemsa stained images/*.png"), k=K_SAMPLES_TRAIN)
    train_masks = [
        path for path in iglob("/content/mseg/Ground truth images/*.png")
        if path.replace("/content/mseg/Ground truth images/", "/content/mseg/Giemsa stained images/").replace("_GT.png", ".png") in train_images
    ]

    return train_images + train_masks

In [9]:
!mkdir -p /content/mseg/train/images
!mkdir -p /content/mseg/train/masks

!mkdir -p /content/mseg/test/images
!mkdir -p /content/mseg/test/masks

In [10]:
train_files = get_mseg_train()

for p in iglob("/content/mseg/Giemsa stained images/*.png"):
    if p in train_files:
        shutil.move(p, p.replace("/content/mseg/Giemsa stained images/", "/content/mseg/train/images/"))
    else:
        shutil.move(p, p.replace("/content/mseg/Giemsa stained images/", "/content/mseg/test/images/"))


for p in iglob("/content/mseg/Ground truth images/*.png"):
    if p in train_files:
        shutil.move(p, p.replace("/content/mseg/Ground truth images/", "/content/mseg/train/masks/"))
    else:
        shutil.move(p, p.replace("/content/mseg/Ground truth images/", "/content/mseg/test/masks/"))

In [11]:
!rm -rf "/content/mseg/Giemsa stained images/"
!rm -rf "/content/mseg/Ground truth images/"

# Data prepare

In [12]:
for p in Path("/content/mseg/train/masks").glob("*.png"):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    bin_mask = np.zeros(img.shape)
    bin_mask[img != 0] = 1
    cv2.imwrite(str(p), bin_mask)

In [13]:
for p in Path("/content/mseg/test/masks").glob("*.png"):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    bin_mask = np.zeros(img.shape)
    bin_mask[img != 0] = 1
    cv2.imwrite(str(p), bin_mask)

# Utils

## Data

In [14]:
def image_mask_path(image_path: str, mask_path: str):
    IMAGE_PATH = Path(image_path)
    IMAGE_PATH_LIST = sorted(list(IMAGE_PATH.glob("*.png")))

    MASK_PATH = Path(mask_path)
    MASK_PATH_LIST = sorted(list(MASK_PATH.glob("*.png")))

    return IMAGE_PATH_LIST, MASK_PATH_LIST

In [15]:
def count_unique(mask_path_list):
    VALUES_UNIQUE_TRAIN = []

    for i in mask_path_list:
        sample = cv2.imread(str(i), cv2.IMREAD_GRAYSCALE)
        uniques = np.unique(sample)
        VALUES_UNIQUE_TRAIN.append(uniques)

    FINAL_VALUES_UNIQUE_TRAIN = np.concatenate(VALUES_UNIQUE_TRAIN)

    return np.unique(FINAL_VALUES_UNIQUE_TRAIN)

In [16]:
class CustomImageMaskDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms, mask_transforms):
        self.data = data
        self.image_transforms = image_transforms
        self.mask_transforms = mask_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")

        state = torch.get_rng_state()
        image = self.image_transforms(image)

        mask_path = self.data.iloc[idx, 1]
        mask = Image.open(mask_path)

        torch.set_rng_state(state)
        mask = self.mask_transforms(mask)

        return image, mask

In [17]:
class CustomTestDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms):
        self.data = data
        self.image_transforms = image_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")
        image = self.image_transforms(image)

        return image

## Train

In [18]:
def train_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer):

    model.train()

    train_loss = 0.
    train_dice = 0.

    for batch, (X,y) in enumerate(dataloader):
        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)
        optimizer.zero_grad()
        logit_mask = model(X)
        loss = loss_fn(logit_mask, y.squeeze())
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp,fp,fn,tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                            target = y.squeeze().cpu().long(),
                                            mode = "multiclass",
                                            num_classes = 21)

        train_dice += smp.metrics.f1_score(tp, fp, fn, tn, reduction = "micro").numpy()

    train_loss = train_loss / len(dataloader)
    train_dice = train_dice / len(dataloader)

    return train_loss, train_dice

In [19]:
def train(model:torch.nn.Module, train_dataloader:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module,
          optimizer:torch.optim.Optimizer, epochs:int = 10):

    results = {'train_loss':[], 'train_dice':[]}

    for epoch in tqdm(range(epochs)):
        train_loss, train_dice = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer)

        print(f'Epoch: {epoch + 1} | ',
              f'Train Loss: {train_loss:.4f} | ',
              f'Train Dice: {train_dice:.4f}')

        results['train_loss'].append(train_loss)
        results['train_dice'].append(train_dice)

    return results

## Prediction

In [20]:
def predictions_mask(model, test_dataloader: torch.utils.data.DataLoader):
    model.eval()

    y_pred_mask = []

    with torch.inference_mode():
        for batch,X in enumerate(test_dataloader):
            X = X.to(device = DEVICE, dtype = torch.float32)
            mask_logit = model(X)
            mask_prob = mask_logit.softmax(dim = 1)
            mask_pred = mask_prob.argmax(dim = 1)
            y_pred_mask.append(mask_pred.detach().cpu())

    y_pred_mask = torch.cat(y_pred_mask)

    return y_pred_mask

# Augmentations

In [21]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

image_transforms = transforms.Compose([
                                       transforms.RandomHorizontalFlip(),
                                       transforms.RandomVerticalFlip(),
                                       transforms.RandomResizedCrop((224, 224), antialias=True),
                                       transforms.ToTensor(),
                                       transforms.Normalize(mean = MEAN, std = STD),
                                       ])

mask_transforms = transforms.Compose([
                                      transforms.RandomHorizontalFlip(),
                                      transforms.RandomVerticalFlip(),
                                      transforms.RandomResizedCrop((224, 224), antialias=True),
                                      transforms.PILToTensor(),
                                    ])

image_transforms_test = transforms.Compose([
                                       transforms.Resize((224, 224), antialias=True),
                                       transforms.ToTensor(),
                                       transforms.Normalize(mean = MEAN, std = STD),
                                       ])

mask_transforms_test = transforms.Compose([
                                      transforms.Resize((224, 224), antialias=True),
                                      transforms.PILToTensor(),
                                    ])

# Data load

In [22]:
image_path_train = "/content/mseg/train/images"
mask_path_train = "/content/mseg/train/masks"

IMAGE_PATH_LIST_TRAIN, MASK_PATH_LIST_TRAIN = image_mask_path(image_path_train,
                                                              mask_path_train)

print(f'Total Images Train: {len(IMAGE_PATH_LIST_TRAIN)}')
print(f'Total Masks Train: {len(MASK_PATH_LIST_TRAIN)}')

Total Images Train: 5
Total Masks Train: 5


In [23]:
print("Unique values Train:")
print(count_unique(MASK_PATH_LIST_TRAIN))

Unique values Train:
[0 1]


# Preprocessing

In [24]:
data_train = pd.DataFrame({'Image': IMAGE_PATH_LIST_TRAIN, 'Mask': MASK_PATH_LIST_TRAIN})

In [25]:
train_dataset = CustomImageMaskDataset(data_train, image_transforms, mask_transforms)

In [26]:
train_dataloader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS)

In [27]:
# We visualize the dimensions of a batch.
batch_images, batch_masks = next(iter(train_dataloader))

batch_images.shape, batch_masks.shape

(torch.Size([5, 3, 224, 224]), torch.Size([5, 1, 224, 224]))

# Model

## Load from checkpoint

In [28]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


You need to add link to this pretrained model ([GDrive](https://drive.google.com/file/d/1-QXEPwwszMdM8LB7FHutwreFWsG4Qpn2/view?usp=drive_link))

In [29]:
model = torch.load("/content/drive/MyDrive/biocad_cis/model.pt", map_location=torch.device('cpu'))

## Freeze layers

In [30]:
model.segmentation_head[0].out_channels = NUM_CLASSES

In [31]:
for param in model.encoder.parameters():
    param.requires_grad = False

In [32]:
# We view our model again to check if the encoder layers freeze.
summary(model = model,
        input_size = [64, 3, 224, 224],
        col_width = 15,
        col_names = ['input_size', 'output_size', 'num_params', 'trainable'],
        row_settings = ['var_names'])

Layer (type (var_name))                            Input Shape     Output Shape    Param #         Trainable
Unet (Unet)                                        [64, 3, 224, 224] [64, 3, 224, 224] --              Partial
├─ResNetEncoder (encoder)                          [64, 3, 224, 224] [64, 3, 224, 224] --              False
│    └─Conv2d (conv1)                              [64, 3, 224, 224] [64, 64, 112, 112] (9,408)         False
│    └─BatchNorm2d (bn1)                           [64, 64, 112, 112] [64, 64, 112, 112] (128)           False
│    └─ReLU (relu)                                 [64, 64, 112, 112] [64, 64, 112, 112] --              --
│    └─MaxPool2d (maxpool)                         [64, 64, 112, 112] [64, 64, 56, 56] --              --
│    └─Sequential (layer1)                         [64, 64, 56, 56] [64, 64, 56, 56] --              False
│    │    └─BasicBlock (0)                         [64, 64, 56, 56] [64, 64, 56, 56] (73,984)        False
│    │    └─BasicBlock

## Train

In [33]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 0.0001)

In [34]:
# Training!!!

SEED = 42
EPOCHS = 10
torch.cuda.manual_seed(SEED)
torch.manual_seed(SEED)

RESULTS = train(model.to(device = DEVICE),
                train_dataloader,
                loss_fn,
                optimizer,
                EPOCHS)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 |  Train Loss: 1.6844 |  Train Dice: 0.3024
Epoch: 2 |  Train Loss: 1.5919 |  Train Dice: 0.3255
Epoch: 3 |  Train Loss: 1.4721 |  Train Dice: 0.3632
Epoch: 4 |  Train Loss: 1.3384 |  Train Dice: 0.4156
Epoch: 5 |  Train Loss: 1.2590 |  Train Dice: 0.5505
Epoch: 6 |  Train Loss: 1.1497 |  Train Dice: 0.4906
Epoch: 7 |  Train Loss: 1.0388 |  Train Dice: 0.5263
Epoch: 8 |  Train Loss: 0.9621 |  Train Dice: 0.5977
Epoch: 9 |  Train Loss: 0.8840 |  Train Dice: 0.6080
Epoch: 10 |  Train Loss: 0.8373 |  Train Dice: 0.6474


## Save

In [35]:
!mkdir /content/checkpoints

In [36]:
torch.save(model.state_dict(), "/content/checkpoints/model.pth")

# Evaluation

In [37]:
image_path_val = "/content/mseg/test/images"
mask_path_val = "/content/mseg/test/masks"

IMAGE_PATH_LIST_VAL, MASK_PATH_LIST_VAL = image_mask_path(image_path_val,
                                                          mask_path_val)

print(f'Total Images Val: {len(IMAGE_PATH_LIST_VAL)}')
print(f'Total Masks Val: {len(MASK_PATH_LIST_VAL)}')

Total Images Val: 878
Total Masks Val: 878


In [38]:
data_val = pd.DataFrame({'Image':IMAGE_PATH_LIST_VAL,
                         'Mask':MASK_PATH_LIST_VAL})
val_dataset = CustomImageMaskDataset(data_val, image_transforms_test,
                                     mask_transforms_test)
val_dataloader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE,
                            shuffle = True, num_workers = NUM_WORKERS)

In [41]:
# Num of batches, which will be used for evaluation
BATCH_TO_TEST = min(30, len(val_dataloader))

In [42]:
test_dice = 0.

with torch.inference_mode():
    for batch, (X, y) in tqdm(enumerate(val_dataloader), total=BATCH_TO_TEST):
        if batch >= BATCH_TO_TEST:
            break

        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)

        logit_mask = model(X)

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp, fp, fn, tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                                target = y.squeeze().cpu().long(),
                                                mode = "multiclass",
                                                num_classes = 3)

        test_dice += smp.metrics.f1_score(tp, fp, fn, tn, reduction = "micro").numpy()

test_dice /= BATCH_TO_TEST

  0%|          | 0/14 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dbec0fa560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79dbec0fa560>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

In [43]:
test_dice

0.7065391966274807